# Dataset2 Annotation with gemma2:27b (Few-Shot)

This notebook annotates dataset2 paragraphs using **gemma2:27b** with **Few-Shot** prompting.

**Best Model Selection**: Based on experimental results:
- **Model**: gemma2:27b
- **Prompt Type**: Few-Shot
- **Entity-Level F1**: 0.7206 (highest among all combinations)
- **Recall**: 0.7197 (captures most entities)
- **Precision**: 0.7215 (fewer false positives)

## Output Format
Produces annotated JSON matching the experiment format:
- `tokens`: List of tokenized words
- `labels`: List of entity labels (PERSON, ORGANIZATION, LOCATION, TIME, CURRENCY, O)
- `text`: Original paragraph text
- `metadata`: Source information (url, title, category, source)

In [6]:
# Cell 1: Setup and Imports
import json
import os
import re
import time
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional

import requests
import numpy as np
from tqdm import tqdm

# Configuration
MODEL_NAME = "gemma2:27b"
OLLAMA_BASE_URL = "http://localhost:11434"
DATASET2_INPUT = "/Users/zeynep_yilmaz/Desktop/ner_nlp/dataset2/selected_140_samples.json"
DATASET2_OUTPUT = "../dataset2/final_annotated.json"
FEW_SHOT_EXAMPLES_PATH = "../data/selected_140_ontonotes5_samples_no_misc.json"  # For few-shot examples

print("✅ Imports and configuration complete")

✅ Imports and configuration complete


In [7]:
# Cell 2: Load Dataset2 Paragraphs
with open(DATASET2_INPUT, 'r', encoding='utf-8') as f:
    dataset2_data = json.load(f)

paragraphs = dataset2_data.get('samples', [])

print(f"✅ Loaded {len(paragraphs)} paragraphs from dataset2")
print(f"Metadata: {dataset2_data.get('metadata', {})}")

# Show example
if len(paragraphs) > 0:
    example = paragraphs[0]
    print(f"\nExample paragraph:")
    print(f"  Text: {example.get('text', '')[:100]}...")
    print(f"  Source: {example.get('source', 'N/A')}")
    print(f"  Category: {example.get('category', 'N/A')}")


✅ Loaded 140 paragraphs from dataset2
Metadata: {'total_samples': 140, 'sources': {'Vice': 70, 'Fox News': 70}, 'selection_date': '2025-12-20T00:35:32.079756', 'selection_strategy': 'Stratified sampling by length, article diversity, and entity richness'}

Example paragraph:
  Text: However, given the state of Ohio’s conservative leanings over the past decade, this man is likely to...
  Source: Vice
  Category: politics


In [8]:
# Cell 3: Load Few-Shot Examples from Training Data
with open(FEW_SHOT_EXAMPLES_PATH, 'r', encoding='utf-8') as f:
    few_shot_dataset = json.load(f)

few_shot_samples = few_shot_dataset.get('samples', [])

print(f"✅ Loaded {len(few_shot_samples)} samples for few-shot examples")

# Select diverse examples (same logic as in experiments)
def select_diverse_few_shot_examples(data: List[Dict], num_examples: int = 5) -> List[Dict]:
    """Select diverse few-shot examples that cover different entity types"""
    samples_with_entities = []
    for sample in data:
        labels = sample['labels']
        entity_counts = {}
        for label in labels:
            if label != 'O':
                entity_counts[label] = entity_counts.get(label, 0) + 1
        samples_with_entities.append({
            'sample': sample,
            'entities': entity_counts,
            'entity_types': set([l for l in labels if l != 'O'])
        })
    
    selected = []
    covered_entity_types = set()
    
    # Prioritize samples with CURRENCY
    currency_samples = [s for s in samples_with_entities if 'CURRENCY' in s['entity_types']]
    if currency_samples:
        selected.append(currency_samples[0]['sample'])
        covered_entity_types.update(currency_samples[0]['entity_types'])
        samples_with_entities.remove(currency_samples[0])
    
    # Prioritize samples with TIME
    time_samples = [s for s in samples_with_entities if 'TIME' in s['entity_types']]
    if time_samples:
        selected.append(time_samples[0]['sample'])
        covered_entity_types.update(time_samples[0]['entity_types'])
        samples_with_entities.remove(time_samples[0])
    
    # Select samples with ORGANIZATION
    org_samples = [s for s in samples_with_entities if 'ORGANIZATION' in s['entity_types']]
    if org_samples:
        selected.append(org_samples[0]['sample'])
        covered_entity_types.update(org_samples[0]['entity_types'])
        samples_with_entities.remove(org_samples[0])
    
    # Fill remaining slots
    remaining_needed = num_examples - len(selected)
    for _ in range(min(remaining_needed, len(samples_with_entities))):
        if not samples_with_entities:
            break
        best_sample = None
        best_score = -1
        for s in samples_with_entities[:20]:
            new_types = s['entity_types'] - covered_entity_types
            score = len(new_types) * 10 + len(s['entity_types'])
            if score > best_score:
                best_score = score
                best_sample = s
        if best_sample:
            selected.append(best_sample['sample'])
            covered_entity_types.update(best_sample['entity_types'])
            samples_with_entities.remove(best_sample)
        else:
            selected.append(samples_with_entities[0]['sample'])
            samples_with_entities.pop(0)
    
    return selected[:num_examples]

few_shot_examples = select_diverse_few_shot_examples(few_shot_samples, num_examples=5)
print(f"\nSelected {len(few_shot_examples)} diverse few-shot examples")
for i, ex in enumerate(few_shot_examples, 1):
    entity_types = set([l for l in ex['labels'] if l != 'O'])
    print(f"  Example {i}: {sorted(entity_types) if entity_types else 'No entities'}")


✅ Loaded 140 samples for few-shot examples

Selected 5 diverse few-shot examples
  Example 1: ['CURRENCY', 'ORGANIZATION', 'TIME']
  Example 2: ['ORGANIZATION', 'TIME']
  Example 3: ['ORGANIZATION']
  Example 4: ['LOCATION']
  Example 5: ['PERSON']


In [9]:
# Cell 4: Define Prompt and Prediction Functions

def extract_entities_from_labels(tokens: List[str], labels: List[str]) -> List[Tuple[str, int, int, str]]:
    """Extract entity spans by grouping consecutive tokens with same entity type"""
    entities = []
    current_entity = None
    
    for i, (token, label) in enumerate(zip(tokens, labels)):
        if label != 'O':
            if current_entity is None:
                current_entity = (label, i, i, token)
            elif current_entity[0] == label:
                entity_type, start, _, text = current_entity
                current_entity = (entity_type, start, i, text + ' ' + token)
            else:
                entities.append(current_entity)
                current_entity = (label, i, i, token)
        else:
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity is not None:
        entities.append(current_entity)
    
    return entities


def create_few_shot_prompt(sentence: str, examples: List[Dict]) -> str:
    """Create few-shot prompt with entity-level JSON output"""
    prompt = """You are an expert Named Entity Recognition system.
Extract ALL entities from sentences using these types only:
- PERSON: People names
- LOCATION: Places (cities, countries, regions)
- ORGANIZATION: Companies, agencies, institutions
- TIME: Dates, years, durations ("2023", "last week", "four years")
- CURRENCY: Money with symbol or name ("$100", "50 million euros", "1.5 billion yen")

Output ONLY a valid JSON array. Format: [{"entity": "exact text span", "type": "TYPE"}, ...]

Here are examples:

"""
    
    for i, example in enumerate(examples, 1):
        text = example.get('text', '')
        tokens = example.get('tokens', [])
        labels = example.get('labels', [])
        
        entities = extract_entities_from_labels(tokens, labels)
        entity_list = [{"entity": entity[3], "type": entity[0]} for entity in entities]
        
        prompt += f"Example {i}:\n"
        prompt += f"Sentence: {text}\n"
        prompt += f"Output: {json.dumps(entity_list)}\n\n"
    
    prompt += f"Now extract entities from this sentence:\n"
    prompt += f"Sentence: {sentence}\n"
    prompt += f"Output:"
    return prompt


def generate_with_ollama(prompt: str, model_name: str = MODEL_NAME, max_retries: int = 3) -> str:
    """Call Ollama API to generate response"""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 1000,
            "temperature": 0.0
        }
    }
    
    for attempt in range(max_retries):
        try:
            if attempt > 0:
                time.sleep(2 ** attempt)  # Exponential backoff
            
            response = requests.post(url, json=payload, timeout=120)
            response.raise_for_status()
            result = response.json()
            return result.get('response', '').strip()
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"⚠️ Connection error (attempt {attempt + 1}/{max_retries}), waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise
    
    return ""


def parse_entities_from_json_response(response: str) -> List[Dict[str, str]]:
    """Parse entities from JSON response"""
    response = response.strip()
    
    json_match = re.search(r'\[.*\]', response, re.DOTALL)
    if json_match:
        json_str = json_match.group(0)
        try:
            entities = json.loads(json_str)
            if isinstance(entities, list):
                valid_entities = []
                for entity in entities:
                    if isinstance(entity, dict) and 'entity' in entity and 'type' in entity:
                        valid_entities.append({
                            'entity': str(entity['entity']).strip(),
                            'type': str(entity['type']).strip().upper()
                        })
                return valid_entities
        except json.JSONDecodeError:
            pass
    
    try:
        entities = json.loads(response)
        if isinstance(entities, list):
            valid_entities = []
            for entity in entities:
                if isinstance(entity, dict) and 'entity' in entity and 'type' in entity:
                    valid_entities.append({
                        'entity': str(entity['entity']).strip(),
                        'type': str(entity['type']).strip().upper()
                    })
            return valid_entities
    except json.JSONDecodeError:
        pass
    
    return []


def tokenize_text(text: str) -> List[str]:
    """Simple tokenization - split on whitespace and handle punctuation"""
    # Split on whitespace
    tokens = text.split()
    # Further split punctuation that's attached to words
    result = []
    for token in tokens:
        # Split punctuation from words (keep punctuation as separate tokens)
        parts = re.findall(r'\w+|[^\w\s]', token)
        result.extend(parts)
    return result


def entities_to_token_labels(entities: List[Dict[str, str]], text: str, tokens: List[str]) -> List[str]:
    """Convert entity spans to token-level labels"""
    labels = ['O'] * len(tokens)
    
    # Create token positions in text
    token_positions = []
    text_lower = text.lower()
    
    start_idx = 0
    for token in tokens:
        pos = text_lower.find(token.lower(), start_idx)
        if pos != -1:
            token_positions.append((pos, pos + len(token)))
            start_idx = pos + len(token)
        else:
            token_positions.append((start_idx, start_idx + len(token)))
            start_idx += len(token) + 1
    
    # Mark entities
    for entity_dict in entities:
        entity_text = entity_dict['entity']
        entity_type = entity_dict['type']
        
        entity_lower = entity_text.lower()
        entity_start = text_lower.find(entity_lower)
        
        if entity_start != -1:
            entity_end = entity_start + len(entity_text)
            
            for i, (tok_start, tok_end) in enumerate(token_positions):
                if (tok_start < entity_end and tok_end > entity_start):
                    labels[i] = entity_type
    
    return labels


def predict_entities(text: str, examples: List[Dict], max_retries: int = 3) -> Tuple[List[str], List[str]]:
    """Predict entities for a text paragraph"""
    # Tokenize
    tokens = tokenize_text(text)
    
    # Create prompt
    prompt = create_few_shot_prompt(text, examples)
    
    # Get model response
    for retry in range(max_retries):
        try:
            response = generate_with_ollama(prompt)
            
            # Parse entities
            entities = parse_entities_from_json_response(response)
            
            if entities:
                labels = entities_to_token_labels(entities, text, tokens)
                return tokens, labels
            
            if retry < max_retries - 1:
                time.sleep(2 ** retry)
                
        except Exception as e:
            if retry < max_retries - 1:
                print(f"⚠️ Prediction error (retry {retry + 1}/{max_retries}): {str(e)[:50]}...")
                time.sleep(2 ** retry)
            else:
                print(f"❌ Prediction failed after {max_retries} retries: {str(e)[:50]}...")
                return tokens, ['O'] * len(tokens)
    
    return tokens, ['O'] * len(tokens)


print("✅ Prediction functions defined")


✅ Prediction functions defined


In [10]:
# Cell 5: Annotate All Paragraphs

print("=" * 80)
print("Annotating Dataset2 with gemma2:27b (Few-Shot)")
print("=" * 80)

annotated_samples = []
errors = []

for idx, para in enumerate(tqdm(paragraphs, desc="Annotating")):
    text = para.get('text', '')
    
    if not text or len(text.strip()) < 10:
        continue
    
    try:
        if idx > 0:
            time.sleep(2.0)  # Delay to prevent overwhelming the model
        
        tokens, labels = predict_entities(text, few_shot_examples)
        
        # Create annotated sample
        annotated_sample = {
            'text': text,
            'tokens': tokens,
            'labels': labels,
            'metadata': {
                'source': para.get('source', ''),
                'url': para.get('url', ''),
                'title': para.get('title', ''),
                'category': para.get('category', '')
            }
        }
        
        annotated_samples.append(annotated_sample)
        
    except Exception as e:
        error_msg = f"Error at paragraph {idx + 1}: {str(e)}"
        errors.append(error_msg)
        print(f"\n❌ {error_msg}")
        continue

print(f"\n✅ Successfully annotated {len(annotated_samples)} paragraphs")
if errors:
    print(f"⚠️ {len(errors)} errors encountered")

# Show statistics
if annotated_samples:
    all_labels = []
    for sample in annotated_samples:
        all_labels.extend(sample['labels'])
    
    label_counts = Counter(all_labels)
    print("\nLabel Distribution:")
    print("=" * 60)
    for label, count in label_counts.most_common():
        pct = count / len(all_labels) * 100 if all_labels else 0
        print(f"{label:15s}: {count:5d} ({pct:5.2f}%)")


Annotating Dataset2 with gemma2:27b (Few-Shot)


Annotating: 100%|██████████| 140/140 [14:03<00:00,  6.03s/it]


✅ Successfully annotated 140 paragraphs

Label Distribution:
O              :  5151 (81.72%)
ORGANIZATION   :   323 ( 5.12%)
PERSON         :   277 ( 4.39%)
TIME           :   259 ( 4.11%)
LOCATION       :   226 ( 3.59%)
CURRENCY       :    66 ( 1.05%)
NUMBER         :     1 ( 0.02%)


In [ ]:
# Cell 6: Save Annotated Dataset

output_data = {
    'metadata': {
        'total_samples': len(annotated_samples),
        'annotation_date': time.strftime('%Y-%m-%dT%H:%M:%S'),
        'model': MODEL_NAME,
        'prompt_type': 'few_shot',
        'few_shot_examples': len(few_shot_examples),
        'original_file': DATASET2_INPUT,
        'annotation_note': 'Annotated using gemma2:27b with Few-Shot prompting (best performing model from experiments)'
    },
    'samples': annotated_samples
}

# Ensure output directory exists
os.makedirs(os.path.dirname(DATASET2_OUTPUT), exist_ok=True)

with open(DATASET2_OUTPUT, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print("=" * 80)
print(f"✅ Annotated dataset saved to: {DATASET2_OUTPUT}")
print(f"   Total samples: {len(annotated_samples)}")
print("=" * 80)

# Show example annotated sample
if annotated_samples:
    print("\nExample annotated sample:")
    example = annotated_samples[0]
    print(f"  Text: {example['text'][:100]}...")
    print(f"  Tokens: {example['tokens'][:10]}...")
    print(f"  Labels: {example['labels'][:10]}...")
    
    # Count entities in example
    entities = [l for l in example['labels'] if l != 'O']
    if entities:
        entity_counts = Counter(entities)
        print(f"  Entities found: {dict(entity_counts)}")


✅ Annotated dataset saved to: ../dataset2/final_annotated.json
   Total samples: 140

Example annotated sample:
  Text: However, given the state of Ohio’s conservative leanings over the past decade, this man is likely to...
  Tokens: ['However', ',', 'given', 'the', 'state', 'of', 'Ohio', '’', 's', 'conservative']...
  Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'LOCATION', 'O', 'O', 'O']...
  Entities found: {'LOCATION': 1, 'TIME': 3}
